In [ ]:
# %%
import re
import json
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

# ------------------------------
# 1. Load Qwen Model
# ------------------------------
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# ------------------------------
# 2. Prompt Template
# ------------------------------
PROMPT = """
You are a clinical information extraction system.

Extract the drug names and adverse drug events (ADEs) from the text below.

Return ONLY valid JSON in this exact format:

{{
  "drug_names": [],
  "adverse_effects": []
}}

Text:
{text}

ONLY RETURN THE JSON. NO EXTRA TEXT.
"""

# ------------------------------
# 3. Helper Functions
# ------------------------------
def trim_text_for_context(text, max_chars=1200):
    """Trim long posts to fit model context window."""
    if len(text) <= max_chars:
        return text
    return text[:max_chars]

def extract_json(response):
    """Extract JSON safely from model output."""
    cleaned = response.replace("```json", "").replace("```", "").strip()
    matches = re.findall(r"\{[\s\S]*?\}", cleaned)
    if matches:
        cleaned = matches[-1]

    try:
        return json.loads(cleaned)
    except Exception as e:
        print("\n⚠️ JSON parse failed:", e)
        print("Raw response:", response)
        print("Cleaned JSON:", cleaned)
        return {"drug_names": [], "adverse_effects": []}

def extract_drug_adr(text):
    trimmed = trim_text_for_context(text)
    prompt = PROMPT.format(text=trimmed)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to("cuda")
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False
    )
    response = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

    # Log responses for debugging
    print("\n--- MODEL RESPONSE START ---")
    print(response[:500] + "..." if len(response) > 500 else response)
    print("--- MODEL RESPONSE END ---\n")

    return extract_json(response)

# ------------------------------
# 4. Load CSV
# ------------------------------
INPUT_CSV = "./data/LLaVA-Med/subset_of_all_ADR.csv"
OUTPUT_CSV = "./data/LLaVA-Med/output_qwen_adr.csv"

df = pd.read_csv(INPUT_CSV)

drug_col = []
adr_col = []

# ------------------------------
# 5. Run Extraction
# ------------------------------
print("\n✅ Starting extraction over CSV...\n")

for text in tqdm(df["Preprocessed Posts"], desc="Extracting ADEs"):
    res = extract_drug_adr(str(text))
    drug_col.append(", ".join(res.get("drug_names", [])))
    adr_col.append(", ".join(res.get("adverse_effects", [])))

df["drug_names"] = drug_col
df["adverse_effects"] = adr_col

# ------------------------------
# 6. Save Output
# ------------------------------
df.to_csv(OUTPUT_CSV, index=False)
print("\n✅ DONE! Output saved to:", OUTPUT_CSV)


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# ------------------------------
# Load the model
# ------------------------------
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # use float16 for GPU efficiency
    device_map="auto"           # automatically put model on GPU if available
)

# ------------------------------
# Quick test prompt
# ------------------------------
prompt = "Extract the drug names and adverse effects from this text: 'Patient experienced nausea after taking ibuprofen.'\n\nReturn JSON only."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # send inputs to GPU

# Generate output
output_tokens = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False
)

# Decode and print
response = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
print("Model output:\n", response)
